# Databricks Workflows & Jobs

Databricks Jobs is the orchestration layer for production workloads. A Job is a named, schedulable unit that runs one or more tasks. Tasks can be notebooks, Python scripts, Delta Live Tables pipelines, SQL queries, dbt projects, or Jar/wheel files.

This notebook covers:
- Job anatomy: tasks, clusters, and dependencies
- Task types and when to use each
- Triggers: schedules, file arrivals, and continuous
- Task parameters and dynamic values
- Retry policies and timeout configuration
- Email alerts and webhook notifications
- Job clusters vs all-purpose clusters
- Multi-task pipeline patterns
- Repair runs and partial re-execution
- Monitoring via the Runs UI and the Jobs API

## 1. Job Anatomy

A Databricks Job consists of:

| Component | Description |
|-----------|-------------|
| **Task** | A single unit of work — one notebook, one pipeline, one script |
| **Task cluster** | The compute assigned to that task (job cluster, existing cluster, or serverless) |
| **Task dependency** | `depends_on` edges that form a DAG |
| **Trigger** | When the job runs (schedule, file arrival, manual) |
| **Parameters** | Key-value pairs passed into the job at trigger time |
| **Notifications** | Email or webhook alerts on success/failure/start |

### Task DAG

Multi-task jobs run as a directed acyclic graph. You specify `depends_on` for each task; Databricks runs independent tasks in parallel and waits for all dependencies before starting a downstream task.

```
ingest_bronze  ──┬──> transform_silver ──> aggregate_gold
                 │
lookup_refresh ──┘
```

In this example `transform_silver` waits for both `ingest_bronze` and `lookup_refresh` to succeed before it starts.

## 2. Task Types

| Task type | Use case |
|-----------|----------|
| **Notebook** | Interactive development, exploratory pipelines |
| **Python script** | Production scripts, packaged logic |
| **Python wheel** | Reusable library installed as a package |
| **Jar** | JVM-based Spark jobs |
| **Delta Live Tables pipeline** | Trigger a DLT pipeline run |
| **SQL** | Databricks SQL queries / dashboards |
| **dbt** | dbt Cloud or Core project runs |
| **Spark submit** | Low-level Spark submit jobs |

**Exam tip:** A DLT pipeline task type triggers a DLT pipeline run — you do **not** embed `import dlt` code in a notebook task and call it from a job. These are separate mechanisms.

## 3. Job Clusters vs All-Purpose Clusters

### Job Cluster (ephemeral)
- Created at job run start, terminated when the run completes
- **Cannot be shared** across jobs or interactive sessions
- Lower cost — no idle time
- Fully reproducible — same config every run
- **Preferred for production jobs**

### All-Purpose Cluster (persistent)
- Shared — multiple users and notebooks can attach to it
- Kept alive between runs (until auto-terminated)
- Faster to start (already running)
- More expensive at scale due to idle time
- **Not recommended for production jobs** — state can bleed between runs

### Serverless Compute (for SQL and notebooks)
- No cluster config required
- Instant startup
- Databricks-managed

**Exam tip:** Each task in a multi-task job can use its own cluster. CPU-heavy tasks can use compute-optimised instances; memory-heavy tasks can use memory-optimised — you configure them independently.

## 4. Triggers

### Scheduled
Quartz cron syntax. Examples:
- `0 0 * * * ?` — every hour
- `0 0 8 * * ?` — daily at 08:00 UTC
- `0 0 8 ? * MON` — every Monday at 08:00

The UI provides a visual cron builder. You can also set a timezone.

### File Arrival (Continuous trigger)
Monitors a cloud storage path and triggers the job when new files land. Useful for event-driven ingestion without a fixed schedule.

### Manual
Run Now button in the UI or `POST /api/2.1/jobs/run-now` via the Jobs API.

### Pause / Unpause
Jobs can be paused without deleting them — useful during deployments or investigations.

**Exam tip:** `availableNow=True` on a Structured Streaming task inside a job is the recommended pattern for incremental batch-style streaming — the stream processes all pending data and stops, allowing the job cluster to terminate.

## 5. Parameters and Dynamic Values

### Job Parameters
Key-value pairs passed to **all tasks** in the job. Available as `dbutils.widgets.get("key")` in notebook tasks or as command-line arguments in Python script tasks.

### Task Parameters
Key-value pairs passed to a **specific task** — override job-level parameters for that task.

### Dynamic Value References
Databricks injects runtime metadata you can reference in parameters:

| Reference | Value |
|-----------|-------|
| `{{job.id}}` | Unique job ID |
| `{{run.id}}` | Unique run ID |
| `{{job.name}}` | Job name |
| `{{task.name}}` | Task key |
| `{{start_time}}` | Run start timestamp (epoch ms) |
| `{{parent_run_id}}` | Parent run ID (for repair runs) |

Use these to build idempotent output paths: e.g. `s3://my-bucket/runs/{{run.id}}/output/`.

## 6. Task Values — Passing Data Between Tasks

`dbutils.jobs.taskValues` lets tasks share small values through the job orchestrator. One task sets a value; a downstream task reads it.

```python
# Task A: ingest_bronze — set a value
row_count = spark.sql("SELECT COUNT(*) FROM bronze.orders").collect()[0][0]
dbutils.jobs.taskValues.set(key="bronze_row_count", value=row_count)
```

```python
# Task B: transform_silver — read the value set by Task A
bronze_count = dbutils.jobs.taskValues.get(
    taskKey="ingest_bronze",
    key="bronze_row_count",
    default=0,
    debugValue=0  # used when running the notebook interactively
)
print(f"Processing {bronze_count} records from Bronze")
```

**Constraints:**
- Values must be JSON-serialisable (string, int, float, bool, list, dict)
- Maximum value size: 48 KB
- Only readable by tasks that `depends_on` the producing task (direct or transitive)

## 7. Retry Policies and Timeouts

### Retry Policy
Configure per task:
- **Max retries** — how many times to retry on failure (0–10)
- **Min retry interval** — minimum wait between retries (seconds)

Retries apply to transient failures (cluster startup failure, OOM, connectivity). They do **not** fix logic errors — a notebook that throws an exception will fail on every retry.

### Timeout
- **Task timeout** — maximum time a single task run can take before being killed
- **Job timeout** — maximum time the entire job run can take

If a task exceeds its timeout, it is marked TIMED_OUT and the job fails (unless the task is configured as optional).

### Optional Tasks
Mark a task as **optional** to allow the job to continue even if that task fails. Downstream tasks that depend on an optional failed task are skipped. Downstream tasks that do **not** depend on it continue normally.

Use optional tasks for non-critical enrichment steps (e.g. sending a notification, refreshing a non-essential cache).

## 8. Notifications and Alerts

### Email Notifications
Configure at job level or task level. Trigger on:
- **On start** — job run begins
- **On success** — job run completes successfully
- **On failure** — job run fails
- **On duration warning** — run exceeds a threshold (useful for SLA monitoring)

### Webhook Notifications
Send an HTTP POST to a URL on the same events. The payload is a JSON object containing the job ID, run ID, job name, and event type. Use this to integrate with PagerDuty, Slack, or custom incident management systems.

### System Tables (Audit Log)
Databricks logs all job run events — start, finish, failure, repair — to system tables in Unity Catalog (`system.lakeflow.job_runs`). You can query these for programmatic monitoring, SLA reporting, and cost attribution.

## 9. Repair Runs

When a multi-task job fails mid-run, you can **repair** the run rather than re-running everything from scratch.

A repair run:
- Keeps the results of all tasks that already succeeded
- Re-runs only the failed task and its downstream dependents
- Uses a new cluster but the same job parameters

This is critical for long-running pipelines where re-running 10 tasks to fix 1 would be wasteful.

**Exam tip:** Repair is only available on multi-task jobs. Single-task jobs must be fully re-run.

You can initiate a repair run from the Runs UI ("Repair Run" button on a failed run) or via `POST /api/2.1/jobs/runs/repair`.

## 10. Multi-Task Pipeline Pattern

A complete medallion architecture job:

```
Task 1: auto_loader_bronze     (Job cluster A — streaming, availableNow)
    |
Task 2: silver_transform       (Job cluster B — batch Spark)
    |
Task 3: gold_aggregates        (Job cluster C — batch Spark)
    |
Task 4: refresh_dashboard      (Serverless SQL — Databricks SQL query)
```

Each task has its own cluster config. Tasks 1–3 are Python notebook tasks. Task 4 is a SQL task that refreshes a Databricks SQL dashboard.

The job is scheduled via cron to run nightly. Task 4 is marked optional so a dashboard refresh failure does not block the pipeline from being marked successful.

### Passing Row Counts Downstream

In [ ]:
# Task 1 — auto_loader_bronze
# Read new files from cloud storage, write to bronze Delta table

checkpoint_path = "/Volumes/main/bronze/checkpoints/orders"
target_table    = "main.bronze.orders"

query = (
    spark.readStream
        .format("cloudFiles")
        .option("cloudFiles.format", "json")
        .option("cloudFiles.schemaLocation", checkpoint_path)
        .load("/Volumes/main/landing/orders/")
        .writeStream
        .format("delta")
        .option("checkpointLocation", checkpoint_path)
        .trigger(availableNow=True)
        .toTable(target_table)
)
query.awaitTermination()

# Pass row count to downstream task
count = spark.table(target_table).count()
dbutils.jobs.taskValues.set(key="bronze_count", value=count)
print(f"Bronze rows: {count}")

In [ ]:
# Task 2 — silver_transform
# Read Task 1's value, apply transformations, write Silver

bronze_count = dbutils.jobs.taskValues.get(
    taskKey="auto_loader_bronze",
    key="bronze_count",
    default=0,
    debugValue=999
)
print(f"Records available from Bronze: {bronze_count}")

if bronze_count == 0:
    print("No new Bronze records — skipping Silver transform")
    dbutils.notebook.exit("no_new_data")

from pyspark.sql.functions import col, to_timestamp, trim

silver_df = (
    spark.readStream
        .format("delta")
        .table("main.bronze.orders")
        .filter(col("order_id").isNotNull())
        .withColumn("order_ts", to_timestamp(col("order_timestamp")))
        .withColumn("customer_id", trim(col("customer_id")))
        .select("order_id", "customer_id", "order_ts", "amount")
)

silver_q = (
    silver_df
        .writeStream
        .format("delta")
        .option("checkpointLocation", "/Volumes/main/silver/checkpoints/orders")
        .trigger(availableNow=True)
        .toTable("main.silver.orders")
)
silver_q.awaitTermination()

## 11. Job Runs API

### Trigger a run programmatically
```bash
curl -X POST https://<workspace-url>/api/2.1/jobs/run-now \
  -H 'Authorization: Bearer <token>' \
  -d '{"job_id": 123, "job_parameters": {"env": "prod", "date": "2025-04-23"}}'
```

### List runs
```bash
curl https://<workspace-url>/api/2.1/jobs/runs/list?job_id=123 \
  -H 'Authorization: Bearer <token>'
```

### Get run output (for notebook tasks)
```bash
curl https://<workspace-url>/api/2.1/jobs/runs/get-output?run_id=456 \
  -H 'Authorization: Bearer <token>'
```

The `notebook_output.result` field in the response contains the last value passed to `dbutils.notebook.exit()`.

**Exam tip:** `dbutils.notebook.exit(value)` sets the notebook task result — readable by the Jobs API and by other notebooks calling `dbutils.notebook.run()`. It does **not** stop the cluster.

## 12. Monitoring Active Jobs

### Runs UI
- **Active runs** tab: real-time task DAG with per-task status
- **Completed runs** tab: full run history with duration, trigger, and outcome
- Click a task node to see its cluster, logs, and error message

### System Tables
Unity Catalog system tables provide historical job run data:

```sql
-- Average run duration per job over the last 30 days
SELECT
  job_id,
  job_name,
  ROUND(AVG(run_duration_seconds) / 60, 1) AS avg_duration_minutes,
  COUNT(*) AS total_runs,
  SUM(CASE WHEN result_state = 'FAILED' THEN 1 ELSE 0 END) AS failed_runs
FROM system.lakeflow.job_runs
WHERE period_start_time >= CURRENT_DATE - INTERVAL 30 DAYS
GROUP BY job_id, job_name
ORDER BY failed_runs DESC
```

**Exam tip:** `system.lakeflow.job_runs` requires Unity Catalog and the `USE` privilege on the `system` catalog.

## 13. Key Exam Topics Summary

| Topic | What to remember |
|-------|------------------|
| Job cluster | Ephemeral, cannot be shared, preferred for production |
| Task DAG | `depends_on` defines edges; independent tasks run in parallel |
| DLT task type | Triggers a DLT pipeline — separate from notebook tasks |
| `availableNow` | Best trigger for streaming tasks inside scheduled jobs |
| `dbutils.jobs.taskValues` | Pass small values between tasks — must be JSON serialisable |
| Repair run | Re-run only failed task + downstream; keep succeeded tasks |
| Optional task | Job continues even if the task fails; dependents are skipped |
| `dbutils.notebook.exit` | Sets task result — readable via Jobs API |
| Timeout | Per-task and per-job; exceeded = TIMED_OUT failure |
| system.lakeflow.job_runs | Historical run data in Unity Catalog system tables |